In [0]:
%pip install faker


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 46.3 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Phase 1 — Synthetic ERP Data Generator (PySpark)
# MAGIC Generates a realistic Procure-to-Pay + General Ledger dataset with **intentionally
# MAGIC injected anomalies** that you will detect in Phase 2 (anomaly detection notebook).
# MAGIC
# MAGIC Tables created (as Delta tables):
# MAGIC - `dim_vendor` — vendor master
# MAGIC - `dim_user` — employee/user master (creators, approvers, buyers)
# MAGIC - `fact_purchase_order` — POs
# MAGIC - `fact_goods_receipt` — goods receipts against POs
# MAGIC - `fact_invoice` — vendor invoices against POs
# MAGIC - `fact_journal_entry` — GL journal entries (manual + system-generated)
# MAGIC
# MAGIC Anomalies injected on purpose (this is your answer key — don't peek during Phase 2!):
# MAGIC 1. ~2% of invoices break 3-way match (amount doesn't tie to PO/GR)
# MAGIC 2. ~1.5% of journal entries have the same user as creator AND approver (SoD conflict)
# MAGIC 3. ~3% of journal entries posted on weekends or after 10pm
# MAGIC 4. A cluster of invoices priced just under the $10,000 approval threshold
# MAGIC 5. An overrepresentation of round-dollar journal entries
# MAGIC 6. A handful of "ghost vendor" entries (vendor flagged high-risk / no matching PO)

# COMMAND ----------

from pyspark.sql import SparkSession, functions as F, types as T
from faker import Faker
import random
from datetime import datetime, timedelta

spark = SparkSession.builder.getOrCreate()
fake = Faker()
Faker.seed(42)
random.seed(42)

# COMMAND ----------

# MAGIC %md ## Config

# COMMAND ----------

N_VENDORS = 150
N_USERS = 60
N_POS = 20000          # purchase orders
APPROVAL_THRESHOLD = 10000
START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2025, 12, 31)

def random_date(start=START_DATE, end=END_DATE):
    delta = end - start
    return start + timedelta(days=random.randint(0, delta.days),
                              hours=random.randint(8, 18),
                              minutes=random.randint(0, 59))

# COMMAND ----------

# MAGIC %md ## dim_vendor

# COMMAND ----------

vendors = []
for i in range(1, N_VENDORS + 1):
    is_high_risk = random.random() < 0.05  # 5% flagged high-risk (shell-company-like)
    vendors.append((
        i,
        fake.company() if not is_high_risk else f"{fake.company()} Trading LLC",
        fake.country(),
        random.choice(["Raw Materials", "IT Services", "Logistics", "Professional Services", "Facilities"]),
        is_high_risk,
        round(random.uniform(0, 1), 2)  # baseline risk score
    ))

vendor_schema = T.StructType([
    T.StructField("vendor_id", T.IntegerType()),
    T.StructField("vendor_name", T.StringType()),
    T.StructField("country", T.StringType()),
    T.StructField("category", T.StringType()),
    T.StructField("is_high_risk_flag", T.BooleanType()),
    T.StructField("baseline_risk_score", T.DoubleType()),
])

dim_vendor = spark.createDataFrame(vendors, schema=vendor_schema)
dim_vendor.write.format("delta").mode("overwrite").saveAsTable("dim_vendor")
display(dim_vendor.limit(5))

# COMMAND ----------

# MAGIC %md ## dim_user

# COMMAND ----------

roles = ["Buyer", "AP Clerk", "GL Accountant", "Controller", "Approver"]
users = []
for i in range(1, N_USERS + 1):
    users.append((i, fake.name(), random.choice(roles), random.choice(
        ["Procurement", "Accounts Payable", "General Ledger", "Finance Controlling"])))

user_schema = T.StructType([
    T.StructField("user_id", T.IntegerType()),
    T.StructField("user_name", T.StringType()),
    T.StructField("role", T.StringType()),
    T.StructField("department", T.StringType()),
])

dim_user = spark.createDataFrame(users, schema=user_schema)
dim_user.write.format("delta").mode("overwrite").saveAsTable("dim_user")
display(dim_user.limit(5))

# COMMAND ----------

# MAGIC %md ## fact_purchase_order

# COMMAND ----------

pos = []
for po_id in range(1, N_POS + 1):
    vendor_id = random.randint(1, N_VENDORS)
    base_amount = round(random.uniform(200, 60000), 2)

    # Anomaly: cluster of amounts just under the approval threshold
    if random.random() < 0.04:
        base_amount = round(random.uniform(9500, 9999), 2)

    po_date = random_date()
    buyer = random.randint(1, N_USERS)
    approver = random.randint(1, N_USERS)
    pos.append((po_id, vendor_id, po_date, base_amount, buyer, approver))

po_schema = T.StructType([
    T.StructField("po_id", T.IntegerType()),
    T.StructField("vendor_id", T.IntegerType()),
    T.StructField("po_date", T.TimestampType()),
    T.StructField("po_amount", T.DoubleType()),
    T.StructField("requested_by", T.IntegerType()),
    T.StructField("approved_by", T.IntegerType()),
])

fact_purchase_order = spark.createDataFrame(pos, schema=po_schema)
fact_purchase_order.write.format("delta").mode("overwrite").saveAsTable("fact_purchase_order")
display(fact_purchase_order.limit(5))

# COMMAND ----------

# MAGIC %md ## fact_goods_receipt
# MAGIC Most POs get a matching goods receipt at ~PO amount. A small % are skipped
# MAGIC entirely (no GR = a control exception you'll flag in Phase 2).

# COMMAND ----------

po_rows = fact_purchase_order.collect()
receipts = []
gr_id = 1
for row in po_rows:
    if random.random() < 0.97:  # 3% of POs never get a goods receipt
        gr_date = row.po_date + timedelta(days=random.randint(1, 20))
        gr_amount = row.po_amount  # normally ties exactly to PO
        receipts.append((gr_id, row.po_id, gr_date, gr_amount, random.randint(1, N_USERS)))
        gr_id += 1

gr_schema = T.StructType([
    T.StructField("gr_id", T.IntegerType()),
    T.StructField("po_id", T.IntegerType()),
    T.StructField("gr_date", T.TimestampType()),
    T.StructField("gr_amount", T.DoubleType()),
    T.StructField("received_by", T.IntegerType()),
])

fact_goods_receipt = spark.createDataFrame(receipts, schema=gr_schema)
fact_goods_receipt.write.format("delta").mode("overwrite").saveAsTable("fact_goods_receipt")
display(fact_goods_receipt.limit(5))

# COMMAND ----------

# MAGIC %md ## fact_invoice
# MAGIC ~2% of invoices deliberately break the 3-way match (amount doesn't tie to
# MAGIC PO/GR — a classic audit exception).

# COMMAND ----------

invoices = []
inv_id = 1
for row in po_rows:
    if random.random() < 0.985:  # a few POs never get invoiced (in-transit / cancelled)
        inv_date = row.po_date + timedelta(days=random.randint(2, 25))
        inv_amount = row.po_amount

        is_mismatch = random.random() < 0.02
        if is_mismatch:
            inv_amount = round(inv_amount * random.uniform(1.05, 1.4), 2)  # overbilled

        invoices.append((inv_id, row.po_id, row.vendor_id, inv_date, inv_amount,
                          random.randint(1, N_USERS), is_mismatch))
        inv_id += 1

inv_schema = T.StructType([
    T.StructField("invoice_id", T.IntegerType()),
    T.StructField("po_id", T.IntegerType()),
    T.StructField("vendor_id", T.IntegerType()),
    T.StructField("invoice_date", T.TimestampType()),
    T.StructField("invoice_amount", T.DoubleType()),
    T.StructField("entered_by", T.IntegerType()),
    T.StructField("_is_planted_mismatch", T.BooleanType()),  # drop this col before Phase 2!
])

fact_invoice = spark.createDataFrame(invoices, schema=inv_schema)
fact_invoice.write.format("delta").mode("overwrite").saveAsTable("fact_invoice")
display(fact_invoice.limit(5))

# COMMAND ----------

# MAGIC %md ## fact_journal_entry
# MAGIC The core dataset for Benford's Law, SoD conflicts, and timing anomalies.

# COMMAND ----------

accounts = ["Revenue", "COGS", "Accrued Liabilities", "Prepaid Expenses",
            "Accounts Payable", "Accounts Receivable", "Payroll Expense", "Other Income"]

journal_entries = []
N_JE = 40000
for je_id in range(1, N_JE + 1):
    dt = random_date()
    amount = round(random.expovariate(1 / 4000), 2)  # right-skewed, natural-looking amounts

    creator = random.randint(1, N_USERS)
    approver = random.randint(1, N_USERS)

    # Anomaly: SoD conflict — same person creates and approves (~1.5%)
    if random.random() < 0.015:
        approver = creator

    # Anomaly: weekend / after-hours postings (~3%)
    if random.random() < 0.03:
        # push into a weekend or late-night slot
        dt = dt.replace(hour=random.choice([23, 0, 1, 2]))
        # nudge to a Saturday
        dt = dt + timedelta(days=(5 - dt.weekday()) % 7)

    # Anomaly: overrepresented round-dollar entries (~4%)
    if random.random() < 0.04:
        amount = float(random.choice([5000, 10000, 15000, 20000, 25000, 50000]))

    source = "System" if random.random() < 0.7 else "Manual"

    journal_entries.append((je_id, dt, random.choice(accounts), amount,
                             creator, approver, source, fake.sentence(nb_words=6)))

je_schema = T.StructType([
    T.StructField("je_id", T.IntegerType()),
    T.StructField("je_datetime", T.TimestampType()),
    T.StructField("account", T.StringType()),
    T.StructField("amount", T.DoubleType()),
    T.StructField("created_by", T.IntegerType()),
    T.StructField("approved_by", T.IntegerType()),
    T.StructField("source", T.StringType()),
    T.StructField("description", T.StringType()),
])

fact_journal_entry = spark.createDataFrame(journal_entries, schema=je_schema)
fact_journal_entry.write.format("delta").mode("overwrite").saveAsTable("fact_journal_entry")
display(fact_journal_entry.limit(5))

# COMMAND ----------

# MAGIC %md ## Sanity check — row counts

# COMMAND ----------

for tbl in ["dim_vendor", "dim_user", "fact_purchase_order",
            "fact_goods_receipt", "fact_invoice", "fact_journal_entry"]:
    cnt = spark.table(tbl).count()
    print(f"{tbl:25s} {cnt:,} rows")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Next: Phase 2
# MAGIC In the next notebook you'll write PySpark logic to **rediscover** the anomalies
# MAGIC planted above — Benford's Law test on `fact_journal_entry.amount`, SoD conflict
# MAGIC detection, weekend/after-hours flagging, 3-way match exceptions between
# MAGIC `fact_purchase_order`, `fact_goods_receipt`, and `fact_invoice`, and an
# MAGIC Isolation Forest outlier score across the journal entry population.
# MAGIC
# MAGIC Drop the `_is_planted_mismatch` column from `fact_invoice` before you start —
# MAGIC it's your answer key, not a real audit signal.


vendor_id,vendor_name,country,category,is_high_risk_flag,baseline_risk_score
1,"Rodriguez, Figueroa and Sanchez",Burundi,Raw Materials,false,0.74
2,"Garcia, Yang and Gardner",Nauru,IT Services,false,0.74
3,Johnson-Davis,Djibouti,Facilities,false,0.09
4,"Guzman, Hoffman and Baldwin",Reunion,Raw Materials,false,0.09
5,"Barnes, Cole and Ramirez",Trinidad and Tobago,Facilities,false,0.03


user_id,user_name,role,department
1,Bradley Smith,Buyer,Procurement
2,Zoe Bell,Controller,Accounts Payable
3,Rachael Pearson,AP Clerk,Finance Controlling
4,Tiffany Gonzalez,Buyer,Accounts Payable
5,Christian Martinez,Buyer,Finance Controlling


po_id,vendor_id,po_date,po_amount,requested_by,approved_by
1,95,2025-02-28T10:19:00.000Z,10233.79,7,38
2,7,2025-07-12T14:45:00.000Z,55747.45,13,5
3,63,2025-12-17T17:51:00.000Z,6293.7,8,51
4,145,2025-08-08T18:23:00.000Z,47001.59,5,33
5,88,2025-09-08T09:27:00.000Z,956.6,24,41


gr_id,po_id,gr_date,gr_amount,received_by
1,1,2025-03-17T10:19:00.000Z,10233.79,4
2,2,2025-07-13T14:45:00.000Z,55747.45,9
3,3,2025-12-20T17:51:00.000Z,6293.7,52
4,4,2025-08-11T18:23:00.000Z,47001.59,28
5,5,2025-09-17T09:27:00.000Z,956.6,30


invoice_id,po_id,vendor_id,invoice_date,invoice_amount,entered_by,_is_planted_mismatch
1,1,95,2025-03-17T10:19:00.000Z,10233.79,9,false
2,3,63,2025-12-20T17:51:00.000Z,6293.7,60,false
3,4,145,2025-08-19T18:23:00.000Z,47001.59,17,false
4,5,88,2025-09-14T09:27:00.000Z,956.6,1,false
5,6,118,2025-10-07T18:17:00.000Z,42497.06,54,false


je_id,je_datetime,account,amount,created_by,approved_by,source,description
1,2025-05-25T10:06:00.000Z,Other Income,3064.95,59,10,System,Training step author coach.
2,2025-09-19T10:36:00.000Z,Accrued Liabilities,803.34,57,13,System,Organization station TV keep light.
3,2025-02-21T16:53:00.000Z,Accounts Receivable,1145.85,48,52,System,Record wall matter management ball.
4,2025-05-09T12:00:00.000Z,Accrued Liabilities,2836.1,44,38,System,Threat same page.
5,2025-05-19T17:49:00.000Z,Accounts Payable,4160.29,33,13,System,Before while structure.


dim_vendor                150 rows
dim_user                  60 rows
fact_purchase_order       20,000 rows
fact_goods_receipt        19,421 rows
fact_invoice              19,707 rows
fact_journal_entry        40,000 rows


In [0]:
%sql
SELECT * FROM fact_journal_entry LIMIT 10

je_id,je_datetime,account,amount,created_by,approved_by,source,description
30001,2025-03-20T17:11:00.000Z,Other Income,4918.69,18,31,System,Memory major off out enjoy.
30002,2025-09-29T12:11:00.000Z,Accounts Payable,6013.55,53,16,System,Staff difficult long especially.
30003,2025-06-11T09:53:00.000Z,Prepaid Expenses,2470.79,19,28,System,Account market stand discover indicate structure.
30004,2025-08-11T09:26:00.000Z,Payroll Expense,1661.93,56,28,System,Alone party thus ago discussion country attention.
30005,2025-09-19T15:14:00.000Z,Prepaid Expenses,1136.85,28,59,System,Figure use industry father network have fear.
30006,2025-11-26T18:49:00.000Z,Accounts Receivable,9745.33,23,2,System,Decision less describe why mother weight now.
30007,2025-07-29T16:34:00.000Z,COGS,10752.03,16,42,System,Summer commercial significant heavy lot energy condition.
30008,2025-09-27T23:58:00.000Z,Accounts Receivable,2710.84,30,56,System,Sell attack among.
30009,2025-09-26T17:33:00.000Z,Other Income,5438.33,38,21,Manual,Name pretty audience baby a speak.
30010,2025-03-10T11:42:00.000Z,Other Income,410.32,58,44,System,Entire authority firm boy figure better foot.
